Ce notebook n'est **pas** fonctionnel. \
C'est un notebook qui a pour vocation d'expliquer comment le modèle fonctionne et est entraîné, il est possible d'essayer de l'entraîner à l'aide du CLI en téléchargeant un dataset dont les liens sont sur le Readme. Cependant l'entraînement peut s'avérer coûteux en ressources, qui est la raison pour laquelle nous nous contentons d'explications ici.

## <u>Le problème : La géolocalisation visuelle</u>

La géolocalisation visuelle consiste à estimer les coordonnées géographiques précises (latitude, longitude) de l'endroit où a été pris l'image à partir d'une seule image. 

Formellement, nous cherchons à apprendre une fonction de mapping $\mathcal{F}$ telle que :
$$\hat{y} = \mathcal{F}(x_{img})$$
Où $x_{img} \in \mathcal{I}$ est une image de requête et $\hat{y} \in \mathbb{R}^2$ est le vecteur de coordonnées estimé.

### Les défis associés
Ce problème est considéré comme très difficile en vision par ordinateur pour plusieurs raisons :

* **La similarité :** Dans un environnement urbain dense comme Paris, de nombreux lieux partagent des caractéristiques sémantiques identiques (architecture haussmannienne, mobilier urbain standardisé, textures de route similaires). Deux rues distinctes peuvent être visuellement **indiscernables** sans un contexte global. C'est un phénomène que l'on retrouve dans tout type d'environnements (ex: forêts, champs, ...).
* **La variabilité d'un lieu :** Une même position géographique peut changer radicalement d'apparence selon l'heure de la journée, la saison, la météo, ou la présence d'objets dynamiques (voitures, piétons) qui créent des occlusions. Le modèle doit apprendre une représentation **invariante** à ces changements.
* **La nature de l'espace :** L'espace géographique est **continu**. Contrairement à un problème de classification classique (ex: "Chat" ou "Chien"), la distinction entre la position $p$ et la position $p + \epsilon$ est infinitésimale.

### Notre approche : Apprentissage contrastif
Pour surmonter ces obstacles, nous formulons le problème comme un apprentissage d'espace métrique (**[Metric Learning](https://tita.lecturer.pens.ac.id/DataScience/M12_13%20-%20DML/M16%20-%20ML%20Deep%20metric%20learning.pdf)**). 
L'objectif n'est **pas** de **prédire** directement une coordonnée, mais d'**aligner** deux espaces vectoriels distincts :
1.  L'espace sémantique de l'image (extrait par **[DinoV2](https://github.com/facebookresearch/dinov2)**).
2.  L'espace topologique de la position (extrait par un **[Neural Field](https://en.wikipedia.org/wiki/Neural_field)**).

Nous cherchons à construire un espace latent commun $\mathcal{Z}$ où la similarité cosinus entre le vecteur image $v_{img}$ et le vecteur position $v_{loc}$ est maximisée si et seulement si l'image a été prise à cette position.

Notre modèle se base originellement sur l'architecture GeoCLIP dont le papier se trouve : [**ici**](https://arxiv.org/abs/2309.16020), mais nous différons sur beaucoup de points. Notamment sur l'utilisation d'images satellites, l'utilisation de DinoV2, notre loss et des adapteurs. \
Nous avons **2** versions du modèle, une qui utilise des images satellites, une autre sans. Nous allons parler **uniquement** de la version avec, la version sans étant analogue. \
Nous avons donc 3 encodeurs :
- Un encodeur de **positions** qui est une fonction $f_{loc} : \mathbb{R}^2 \rightarrow \mathcal{Z}$
- Un encodeur d'**images** qui est une fonction $f_{img} : \mathcal{I} \rightarrow \mathcal{Z}$
- Un encodeur d'**images satellites** qui est une fonction $f_{sat} : \mathcal{I}_{sat} \rightarrow \mathcal{Z}$ 

Ici $f_{img}$ et $f_{sat}$ partagent la même architecture, avec des poids distincts. 

L'encodeur dédié aux images satellites étant surtout présent en tant que "distillateur" de connaissances, pour faire correspondre les images et donner implicitement plus de contexte.\

### Quelques remarques :
- **Pourquoi DinoV2 ?** : La tâche de la géolocalisation étant remarquablement **difficile** nous avons fait le choix d'utiliser un modèle de fondation, c'est à dire un modèle pré-entraîné que nous pouvons ensuite utiliser pour nos tâches. Le modèle de fondation ici est DinoV2 en sa version "Small", que nous utilisons en tant qu'extracteur. 

- **Pourquoi pas de CNN ?** : Comme DinoV2 s'occupe de récupérer les features visuelles, nous travaillons soit avec des features RFF soit avec des features sortant de DinoV2, l'utilisation de CNNs nécessiterait une discrétisation pour assez peu de résultats.

- **Et pourquoi pas de Transformers ?** Pour l'Image Encoder, Dino est déjà supposé faire le gros du travail donc il est dans notre intérêt de rester léger. Pour le Location Encoder la principale raison et que nous utilisons un [Neural Field](https://kaldir.vc.in.tum.de/adl4cv/ws2425/3.NeuralFields.pdf) qui diffère des architectures classiques et qui est naturellement conçue pour justement travailler et comprendre le biais induit par les positions, ce qui manque aux transformers.


### <u>Partie 0 : Le dataset</u>

Tous nos datasets sont issus de **Google Streeview** (images collectées à l'aide de [**streetlevel**](https://github.com/sk-zk/streetlevel), qui sont des panoramas que l'on découpe).

Nous avons 3 datasets :
- France "Urbaine", 70K images pour 35K positions.
- France "Uniforme", contient 300K images pour 150K positions.
  - Sans métadonnées, ce dataset n'est à notre avis pas complètement exploitable. Mais c'est une bonne base que nous n'avons pas eu le temps de raffiner.
- Un dataset concentré sur la région Parisienne proche (La Defense/Cachan sont les limites à l'Ouest et au Sud par ex). Il a également en complément, pour chaque image, son image satellite associée (qui couvre 300mx300m).

Nous définissons 2 tâches **distinctes**, qui émergent en fonction du découpage du dataset :
- La compréhension de zones nouvelles (zero-shot)
  - Nécessite de découper le dataset en grille, distribuant les cases entre set d'entraînement et de validation.
  - Note : Nous avons trouvé cette tâche plus dure que la version normale du problème, qui est déjà dure.

- L'interpolation entre des positions proches déjà vues 
  - Découpage aléatoire classique entre train set et validation set.
  - Note : Il faut faire attention à prendre toutes les images d'un même panorama pour limiter la fuite de données.


## <u>Partie 1 : le modèle</u>

Avant toute chose, voici un schéma résumant le modèle et son processus d'apprentissage : 
\
<img src="https://cdn.discordapp.com/attachments/997413615290302536/1466430356625293334/mermaid-diagram-2026-01-29-145019.png?ex=697cb731&is=697b65b1&hm=f5e1dad4832a1909d21eea4559f50d4efb6a910f9836efd82657783c7a7649f3&" alt="drawing" width="1200"/>

### <u>**L'encodeur d'images** (ImageEncoder)</u>

L'encodeur d'images contient deux parties :
- Le **backbone** et les **adaptateurs** :
  - L'on utilise [**DinoV2**](https://github.com/facebookresearch/dinov2) gelé pour extraire les features de l'image, on récupère :
    - cls_token qui capture l'information sémantique globale. 
    - patch_tokens qui capture les détails spatiaux locaux.
  - Deux têtes **d'adapteurs**, une pour chaque type de feature que l'on récupère de DinoV2
    - Leur objectif est d'adapter la sortie brute de DinoV2 en de la donnée plus facilement compréhensible pour la projection.
    - L'idée des adapteurs est inspiré du papier [PEFT](https://arxiv.org/pdf/1902.00751/1000). 
  - On applique également une couche de [GeM Pooling](https://amaarora.github.io/posts/2020-08-30-gempool.html), qui est un équilibre entre le max pooling et l'avg pooling, permettant alors de réduire les dimensions intelligemment.
- On utilise une tête de **projection**, un MLP à 2 couches cachées.


#### Cet encodeur est utilisé pour les images StreetView ET pour les images satellites !

Le code des adapteurs :
```python
class FeatureAdapter(nn.Module):
    def __init__(self, in_dim=384, bottleneck=96):
        super().__init__()
        self.down = nn.Linear(in_dim, bottleneck)
        self.act = nn.GELU()
        self.up = nn.Linear(bottleneck, in_dim)
        self.scale = nn.Parameter(torch.ones(1) * 0.1)
    
    def forward(self, x):
        out = self.down(x)
        out = self.act(out)
        out = self.up(out)
        return x + self.scale * out
```

Le code de la projection : 
```python
self.proj = nn.Sequential(
    nn.Linear(768, 2048),
    nn.LayerNorm(2048),
    nn.GELU(),
    #On a très peu de données, donc on utilise du dropout pour éviter l'overfitting
    nn.Dropout(0.3),
    nn.Linear(2048, 1024),
    nn.LayerNorm(1024),
    nn.GELU(),
    nn.Dropout(0.3),
    nn.Linear(1024, 512)
)
```

La forward se passe alors comme :
```python
#On applique dino sur notre image, on récupère les features
with torch.no_grad():
    output = self.backbone.forward_features(x)

#On récupère les deux types de features
patch_tokens = self.patch_adapter(output["x_norm_patchtokens"])
cls_token = self.cls_adapter(output["x_norm_clstoken"])

#On applique le GeM pooling
pooled_patches = self.pool(patch_tokens)

#L'on combine pour pouvoir projeter !
combined = torch.cat([pooled_patches, cls_token], dim=1)

embeddings = self.proj(combined)
return F.normalize(embeddings, p=2, dim=1)
```

### <u>**L'encodeur de positions** (Location Encoder)</u>
L'encodeur de positions est plus simple mais a plus de paramètres, il est très inspiré de l'architecture GeoCLIP, qui est lui même une architecture classique. \
La pipeline suit 4 étapes : 
- On normalise les données, qu'elles soient dans [-1,1] est essentiel pour appliquer extraire les RFF
  - Dans un contexte **global** il faudrait projeter les coordonnées avant de normaliser pour prendre un compte la courbature de la Terre.
- On encode les positions à l'aide des [**Random Fourier Features** (RFF)](https://github.com/jmclong/random-fourier-features-pytorch), ce qui aide à contrer le [**biais spectral**](https://en.wikipedia.org/wiki/Frequency_principle/spectral_bias) (apprentissage de détails grossiers, passant au dessus de subtilités). 
    - Nous utilisons plusieurs échelles d'écarts-types $\sigma$ :
      - Les **petits** $\sigma$ aident à capter la structure **globale**
      - Les **grands** $\sigma$ aident à capturer des détails **fins**
    - Il existe aussi d'autres alternatives plus modernes comme [SIREN](https://arxiv.org/pdf/2006.09661) ou même [un mélange des deux](https://xeonqq.github.io/machine%20learning/fourier-feature-siren/). Cependant nous n'avons pas réussi à faire converger un modèle utilisant SIREN.
- Nous traitons ensuite **hiérarchiquement** les différents $\sigma$, en les passant à travail un MLP résiduel, permettant de progressivement affiner l'information.
- On projette finalement vers une dimension de 512 en une couche pleinement connectée

Le code des blocs résiduels est :
```python
class ResBlock(nn.Module):
    def __init__(self, hidden_dim, encoded_size=256, dropout=0.1):
        super().__init__()
        self.rff_proj = nn.Linear(encoded_size * 2, hidden_dim)
        self.net = nn.Sequential(
            nn.LayerNorm(hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, hidden_dim),
            nn.Dropout(dropout),
            nn.Linear(hidden_dim, hidden_dim) 
        )
    def forward(self, current_state, rff_feat):
        freq_emb = self.rff_proj(rff_feat)
        x = current_state + freq_emb
        out = self.net(x)
        return x + out
```

Le location encoder est alors :
```python
self.sigmas = [2.0, 16.0, 64.0, 256.0]
#Dans la version cross-view spécialisée sur Paris nous avons choisi : [1, 2, 4, 8, 16, 32, 64, 128]

encoded_size = 256
hidden_dim = 1024 

self.rff_layers = nn.ModuleList([
    rff.layers.GaussianEncoding(sigma=s, input_size=2, encoded_size=encoded_size)
    for s in self.sigmas
])

self.blocks = nn.ModuleList([
    ResBlock(hidden_dim, encoded_size, dropout=0.1)
    for _ in self.sigmas
])

#On laisse au modèle choisir le point de départ pour l'inférence
self.start_token = nn.Parameter(torch.randn(1, hidden_dim))
self.final_proj = nn.Linear(hidden_dim, 512)
```

Est la forward est alors : 
```python
x = normalize(x)
state = self.start_token.expand(x.shape[0], -1)
for i, layer in enumerate(self.rff_layers):
    rff_feat = layer(x)
    state = self.blocks[i](state, rff_feat)
x = self.final_proj(state)
return F.normalize(x, p=2, dim=1)
```

### <u>**L'encodeur global**</u>
Ce n'est alors qu'un encapsuleur des deux encodeurs précédents avec un paramètre de scale utilisé dans la loss, permettant d'aider l'apprentissage.
```python
class MixedEncoder(nn.Module):
    def __init__(self):
        super(MixedEncoder, self).__init__()
        self.image_encoder = ImageEncoder()
        self.location_encoder = LocationEncoder()
        self.sat_encoder = ImageEncoder()
        #2.6592 = logit scale classique pour CLIP loss
        self.logit_scale = nn.Parameter(torch.ones([]) * 2.6592)

    def forward(self, img, loc, sat):
        img_embed = self.image_encoder(img)
        loc_embed = self.location_encoder(loc)
        sat_embed = self.sat_encoder(sat)
        return img_embed, loc_embed, sat_embed, self.logit_scale.exp()
```

## <u> **Partie 2 : l'apprentissage** </u>

Notre modèle repose sur l'**apprentissage contrastif**, nous avons des triplets image/position/sat et nous cherchons à les faire correspondre dans un espace latent commun.
 - Note : Nous avons également essayé d'utiliser une triplet margin loss, sans convergence satisfaisante en raison de la difficulté à avoir des triplets pertinents.

### Les données lors de l'apprentissage

Nous devons alors retourner des couples img/position, nous utilisons des augmentations sur les images afin de s'assurer que le modèle ne se focus pas sur des détails tels que la saison et nous ajoutons également un (léger) bruit aux coordonnées lors de l'entraînement. Dans l'espoir d'obtenir une meilleur robustesse.

### La loss

Soit $M_{i,j}$ la matrice des dupliqués (à epsilon près), une case est à $1$ si $i = j$ ou si la position $i$ est à distance epsilon de la position $j$. \
Plus formellement cela s'écrit : 
$
M_{i,j} = 
    \begin{cases} 
        1 & \text{si } i = j \\
        0 & \text{si } i \neq j \text{ et } \|\text{pos}_i - \text{pos}_j\| < \epsilon \\
        1 & \text{sinon}
    \end{cases} $ \
On définit alors la [Loss InfoNCE](https://lilianweng.github.io/posts/2021-05-31-contrastive/#infonce) masquée (c'est à dire prenant compte des doublons à l'aide de la matrice $M$) entre deux types d'entrées (img, sat ou loc) comme : $$\mathcal{L}_{\text{NCE}}(\mathbf{X}, \mathbf{Y}) = - \frac{1}{N} \sum_{i=1}^N \log \left( \frac{f(\mathbf{x}_i, \mathbf{y}_i)}{\sum_{j=1}^N M_{i,j} \cdot f(\mathbf{x}_i, \mathbf{y}_j)} \right)$$ 
Où $f$ est ici l'exponentielle du score de similarité ajustée au logit_scale. \
L'objectif du masquage est de ne pas pénaliser le modèle si il confond deux images prises à quelques mètres l'une de l'autre, car elles sont toutes les deux sémantiquement valides. \
On peut alors définir la loss symétrique : $$\mathcal{L}_{\text{sym}}(\mathbf{X}, \mathbf{Y}) = \frac{1}{2} \left[ \mathcal{L}_{\text{NCE}}(\mathbf{X}, \mathbf{Y}) + \mathcal{L}_{\text{NCE}}(\mathbf{Y}, \mathbf{X}) \right]$$ \
Et l'on a alors une loss symétrique pour chaque paire de types d'entrées, on alors : 
    $$\mathcal{L}_{\text{total}} = \lambda_1 \mathcal{L}_{\text{sym}}(\mathbf{v}^{\text{img}}, \mathbf{v}^{\text{loc}}) 
    + \lambda_2 \mathcal{L}_{\text{sym}}(\mathbf{v}^{\text{img}}, \mathbf{v}^{\text{sat}}) 
    + \lambda_3 \mathcal{L}_{\text{sym}}(\mathbf{v}^{\text{sat}}, \mathbf{v}^{\text{loc}})$$ \
Où les $\lambda_i$ sont les différents coefficients (ici généralement $(0.4,0.4,0.2)$)

L'intuition derrière cette loss est qu'elle va récompenser les bons guess X <-> Y mais punir les mauvais. Cette loss est très dépendante de la taille des batch, certains modèles CLIP vont jusqu'à des batch de taille 32000. \
\
Mais en raison de contraintes matérielles nous une utilisation **accumulation d'embeddings** :
- Nous utilisons des plus petits batch (32 généralement)
- Nous accumulons les résultats des petits batchs **sans faire de loss ou de backward** tant que l'on a moins de 1024 éléments
- Nous calculons la loss sur les 1024 accumulés et nous faisons enfin une mise à jour du réseau.

Ce qui nous permet de profiter de la loss constrastive en simulant des grands batchs à plus petits coût, au prix d'une actualisation tous les 32 batches.
\
Il existe aussi d'autres méthodes (des heuristiques) pour gérer le besoin de gros batches de la loss contrastive, mais comme nous avons un modèle simple et que DinoV2 est gelé, nous pouvons nous permettre de ne pas utiliser d'heuristique.

## <u> **Partie 3 : Le retrieval** </u>
Une fois le modèle entraîné, il faut créer une base de données pour ensuite pouvoir faire du retrieval. \
Il existe plusieurs méthodes :
- Utiliser une grille (que l'on encode à l'aide de $f_{pos}$)
- Faire un sampling du dataset.

Nous utilisons un sampling du dataset ici. Et nous appliquons un algorithme de [k-Nearest Neighbors](https://en.wikipedia.org/wiki/K-nearest_neighbors_algorithm).
- Note : L'algorithme de k-NN étant linéaire en haute dimension, nous utilisions une structure de KMeans hiérarchique, mais étant une structure probabiliste et étant donné que nous avons assez peu de données, nous l'avons supprimée.